In [ ]:
from langgraph.graph import StateGraph,START, END
from typing import TypedDict, Literal, Annotated
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.messages import SystemMessage, HumanMessage
import operator

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()

hf_token = os.getenv("HUGGINGFACEHUB_API_TOKEN")

In [ ]:
base_generator_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=hf_token,
    temperature=0.7
)
generator_llm = ChatHuggingFace(llm=base_generator_llm)

base_evaluator_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=hf_token,
    temperature=0.2
)
evaluator_llm = ChatHuggingFace(llm=base_evaluator_llm)

base_optimizer_llm = HuggingFaceEndpoint(
    repo_id="Qwen/Qwen2.5-7B-Instruct",
    huggingfacehub_api_token=hf_token,
    temperature=0.5
)
optimizer_llm = ChatHuggingFace(llm=base_optimizer_llm)

In [ ]:
from pydantic import BaseModel, Field

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs_improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [ ]:
# Use manual parsing since model returns structured text, not JSON
import json
import re

def _manual_structured_invoke(messages):
    """Parse structured text or JSON from model output"""
    resp = evaluator_llm.invoke(messages)
    text = getattr(resp, "content", str(resp))
    
    # Try JSON first
    try:
        if "```json" in text:
            json_text = text.split("```json")[1].split("```")[0]
        elif "```" in text:
            json_text = text.split("```")[1].split("```")[0]
        else:
            json_text = text
        parsed = json.loads(json_text)
        return parsed
    except (json.JSONDecodeError, IndexError):
        pass
    
    # Try parsing structured text format: "evaluation: ...\n\nfeedback: ..."
    try:
        result = {}
        
        # Extract evaluation
        eval_match = re.search(r'evaluation:\s*([^\n]+)', text, re.IGNORECASE)
        if eval_match:
            result['evaluation'] = eval_match.group(1).strip().lower()
        
        # Extract feedback
        feedback_match = re.search(r'feedback:\s*(.+?)(?=\n\n|$)', text, re.IGNORECASE | re.DOTALL)
        if feedback_match:
            result['feedback'] = feedback_match.group(1).strip()
        
        if result:
            return result
    except Exception:
        pass
    
    # Fallback: treat entire response as feedback with default evaluation
    return {
        'evaluation': 'needs_improvement',
        'feedback': text
    }

# Create wrapper that has an invoke method
class ManualStructuredEvaluator:
    def invoke(self, messages):
        return _manual_structured_invoke(messages)

structured_evaluator_llm = ManualStructuredEvaluator()


In [ ]:
# structured_evaluator_llm = evaluator_llm.with_structured_output(TweetEvaluation,)

In [ ]:
#state
class TweetState(TypedDict):
    topic : str
    tweet : str
    evaluation : Literal['approved','need_improvement']
    feedback : str
    iteration : int
    max_iteration : int
    tweet_history : Annotated[list[str], operator.add]
    feedback_history : Annotated[list[str], operator.add]


In [ ]:

def generate_tweet(state: TweetState):

    # prompt
    messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english
""")
    ]

    # send generator_llm
    response = generator_llm.invoke(messages).content

    # return response
    return {'tweet': response, 'tweet_history': [response]}

In [ ]:

def evaluate_tweet(state: TweetState):

    # prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]

    response = structured_evaluator_llm.invoke(messages)
    if isinstance(response, dict):
        evaluation = response.get('evaluation', 'needs_improvement')
        feedback = response.get('feedback', '')
    else:
        evaluation = getattr(response, 'evaluation', 'needs_improvement')
        feedback = getattr(response, 'feedback', '')

    return {'evaluation': evaluation, 'feedback': feedback, 'feedback_history': [feedback]}


In [ ]:

def optimize_tweet(state: TweetState):

    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]

    response = optimizer_llm.invoke(messages).content
    iteration = state['iteration'] + 1

    return {'tweet': response, 'iteration': iteration, 'tweet_history': [response]}


In [ ]:

def route_evaluation(state: TweetState):

    if state['evaluation'] == 'approved' or state['iteration'] >= state['max_iteration']:
        return 'approved'
    else:
        return 'needs_improvement'

In [ ]:

graph = StateGraph(TweetState)

graph.add_node('generate', generate_tweet)
graph.add_node('evaluate', evaluate_tweet)
graph.add_node('optimize', optimize_tweet)


graph.add_edge(START, 'generate')
graph.add_edge('generate', 'evaluate')
graph.add_conditional_edges('evaluate', route_evaluation, {'approved': END, 'needs_improvement': 'optimize'})
graph.add_edge('optimize', 'evaluate')

workflow = graph.compile()

In [ ]:
workflow

In [ ]:
initial_state = {
    "topic": "srhberhb",
    "iteration": 1,
    "max_iteration": 5
}
result = workflow.invoke(initial_state)


In [ ]:
result

In [ ]:
for tweet in result['tweet_history']:
    print(tweet)